In [ ]:
!pip install pandas torch transformers scikit-learn numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import random
import os

# Фиксация seed для воспроизводимости
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

# Настройки
MODEL_NAME = 'cointegrated/rubert-tiny2' # Легкая и быстрая модель для русского языка
MAX_LEN = 128   # Максимальная длина текста в токенах
BATCH_SIZE = 32 # Размер батча
EPOCHS = 3      # Количество эпох
LEARNING_RATE = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# --- 1. Подготовка данных ---

class ToxicDataset(Dataset):
    def __init__(self, texts, targets=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        output = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

        if self.targets is not None:
            output['targets'] = torch.tensor(self.targets[item], dtype=torch.float)

        return output

# Чтение данных
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
except FileNotFoundError:
    print("Ошибка: Файлы train.csv или test.csv не найдены.")
    exit()

# Очистка пустых значений
train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

# Разделение на train и validation
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'].values,
    train_df['score'].values,
    test_size=0.2,
    random_state=42
)

# Инициализация токенизатора
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Создание датасетов
train_dataset = ToxicDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset = ToxicDataset(X_val, y_val, tokenizer, MAX_LEN)
test_dataset = ToxicDataset(test_df['text'].values, None, tokenizer, MAX_LEN)

# Создание загрузчиков данных
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# --- 2. Инициализация модели ---

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1, # Бинарная классификация -> 1 выходной нейрон
    problem_type="multi_label_classification" # Для использования BCE loss внутри модели
)
model = model.to(DEVICE)

# Оптимизатор
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# Функция потерь
loss_fn = torch.nn.BCEWithLogitsLoss()

In [ ]:
# --- 3. Цикл обучения ---

def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    losses = []

    for d in tqdm(data_loader, desc="Обучение"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["targets"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Получаем логиты и сжимаем размерность
        logits = outputs.logits.squeeze(-1)

        loss = loss_fn(logits, targets)
        losses.append(loss.item())

        loss.backward()
        optimizer.step()

    return np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    predictions = []
    real_values = []

    with torch.no_grad():
        for d in tqdm(data_loader, desc="Валидация"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["targets"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # Применяем сигмоиду для получения вероятности [0, 1]
            probs = torch.sigmoid(outputs.logits.squeeze(-1))

            predictions.extend(probs.cpu().numpy())
            real_values.extend(targets.cpu().numpy())

    try:
        roc_auc = roc_auc_score(real_values, predictions)
    except ValueError:
        roc_auc = 0.0 # Если в батче только один класс

    return roc_auc

# Запуск обучения
print("Начинаем обучение...")
for epoch in range(EPOCHS):
    print(f'Эпоха {epoch + 1}/{EPOCHS}')
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, DEVICE)
    val_roc_auc = eval_model(model, val_loader, DEVICE)

    print(f'Train Loss: {train_loss:.4f}')
    print(f'Val ROC AUC: {val_roc_auc:.4f}')

In [ ]:
# --- 4. Предсказание на тесте ---
print("Генерация предсказаний для test.csv...")
model.eval()
test_probs = []

device = DEVICE
with torch.no_grad():
    for d in tqdm(test_loader, desc="Тестирование"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.sigmoid(outputs.logits.squeeze(-1))
        test_probs.extend(probs.cpu().numpy())


In [ ]:
# --- 5. Формирование submission.csv ---

submission = pd.DataFrame({
    'id': test_df['id'],
    'score': test_probs
})

submission.to_csv('submission.csv', index=False)